In [0]:
sales_df = spark.read.table('de_mini_project.azure_blob_storage.sales')
display(sales_df)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType
sales_df = sales_df.drop("_file","_line","_modified","_fivetran_synced")
sales_df = sales_df.withColumnRenamed("Trxn_ID#","transaction_id").withColumnRenamed("PROD_CODE_ID","sku_id").withColumnRenamed("Cust_ID_99","customer_id").withColumnRenamed("Store_Loc_ID","store_id").withColumnRenamed("Qty_Sold","qty_sold").withColumnRenamed("_Unit_Price_","unit_price").withColumnRenamed("!Date_Ref!","date")   
sales_df = sales_df.withColumn("date",F.coalesce(
    F.try_to_date(F.col("date"), "dd-MM-yyyy"),
    F.try_to_date(F.col("date"), "MM-dd-yyyy"),
    F.try_to_date(F.col("date"), "dd-MMM-yy"),
    F.try_to_date(F.col("date"), "yyyy.MM.dd"),
    F.try_to_date(F.col("date"), "MMM dd, yyyy"),
    F.try_to_date(F.col("date"), "yyyyMMdd"),
))
sales_df = sales_df.withColumn("unit_price",F.regexp_replace("unit_price",'INR',''))
sales_df = sales_df.withColumn("unit_price",F.regexp_replace("unit_price", '\\$', ''))
sales_df = sales_df.withColumn("unit_price",F.regexp_replace("unit_price"," ",""))
sales_df = sales_df.withColumn("unit_price", F.col("unit_price").cast(DecimalType()))

sales_df = sales_df.replace("NULL","Unkown Customer")
display(sales_df)

In [0]:
sales_df.write.format("delta").mode("overwrite").saveAsTable("de_mini_project.silver.sales")